# CRYCHIC developer tutorial: multi-condition subject cross-fit

This notebook is a self-contained path from a multi-condition/group `AnnData` object to persisted, queryable CRYCHIC cross-fit diagnostics. It uses a tiny synthetic dataset by default and can be pointed at a real H5AD and checksum-pinned resources without changing the analysis cells.

CRYCHIC is **not a deep-learning model**. Outer and inner folds are subject-blocked statistical resampling scopes, not neural-network train/validation sets. The current release emits descriptive held-out component ledgers, subject-family effects, repeated split-stability diagnostics, and auditable full-pipeline resample records. It does **not** release calibrated p-values, q-values, confidence intervals, `comm_probability`, posterior probabilities, causal sender claims, or a `method_superiority` claim.


In [ ]:
from __future__ import annotations

import shutil
from pathlib import Path

import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import sparse

import crychic
from crychic.attribution import DirectionalContrastPairSpec, PenaltyTuningSpec
from crychic.design import balanced_contrast
from crychic.resources import (
    GeneNamespace,
    Interaction,
    MappingReport,
    ResourceBundle,
    Species,
    TargetPrior,
)
from crychic.sender import ContrastCommonSenderParameters

USE_SYNTHETIC = True
REAL_H5AD = Path("/path/to/cohort.h5ad")
DATABASE_ROOT = Path("/path/to/checksum_pinned_databases")
LR_RESOURCE = "cellchat"  # or "cellphonedb"
OUTPUT_ROOT = Path("tutorial_output/developer_subject_crossfit")
OVERWRITE_OUTPUT = True
RUN_REPEATED = False
RUN_RESAMPLING = False

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print(f"CRYCHIC {crychic.__version__}; output={OUTPUT_ROOT.resolve()}")

## 1. Input contract and tiny multi-group H5AD

Required `.obs` roles are distinct: `sample_id` is the library/sampling unit, `subject_id` is the independent biological unit, `cell_type` identifies sender/receiver populations, and every field in `context_keys` defines a condition/group axis. A subject's samples stay together in outer folds, bootstrap draws, and legal permutation operations. Raw integer-like counts belong in `adata.layers['counts']`; cells are aggregated into sample-level evidence and are never treated as independent replicates.


In [ ]:
def make_tiny_multigroup_adata(n_subjects: int = 8) -> ad.AnnData:
    genes = ("L1", "R1", "L2", "R2", "T1", "T2")
    rows: list[list[int]] = []
    obs_rows: list[dict[str, str]] = []
    obs_names: list[str] = []
    for subject_index in range(n_subjects):
        subject = f"p{subject_index + 1}"
        for condition_index, condition in enumerate(("control", "stim")):
            sample = f"{subject}-{condition}"
            signal = 40 + 3 * subject_index + 20 * condition_index
            background = 4 + subject_index
            profiles = {
                "Sender": [signal, 0, background, 0, 1, 1],
                "Receiver": [
                    0,
                    signal,
                    0,
                    background,
                    6 + 8 * condition_index,
                    3 + 4 * condition_index,
                ],
            }
            for cell_type, profile in profiles.items():
                for cell_index in range(3):
                    rows.append(profile)
                    obs_rows.append(
                        {
                            "sample_id": sample,
                            "subject_id": subject,
                            "cell_type": cell_type,
                            "condition": condition,
                        }
                    )
                    obs_names.append(f"{sample}-{cell_type}-{cell_index}")
    counts = sparse.csr_matrix(np.asarray(rows, dtype=np.int64))
    result = ad.AnnData(
        X=sparse.csr_matrix(counts.shape, dtype=np.float64),
        obs=pd.DataFrame(obs_rows, index=obs_names),
        var=pd.DataFrame(index=genes),
    )
    result.layers["counts"] = counts
    return result


adata = make_tiny_multigroup_adata() if USE_SYNTHETIC else ad.read_h5ad(REAL_H5AD)
adata

In [ ]:
config = crychic.CrychicConfig(
    context_keys=("condition",),
    counts_layer="counts",
    sample_key="sample_id",
    subject_key="subject_id",
    cell_type_key="cell_type",
    design="~ condition",
    random_seed=20260715,
)
validator = crychic.Crychic(config)
validated = validator.validate(adata)
validation_summary = {
    "mode": validated.report.mode.value,
    "n_cells": validated.report.n_obs,
    "n_genes": validated.report.n_vars,
    "n_samples": validated.report.n_samples,
    "n_subjects": validated.report.n_subjects,
    "input_inference_eligible": validated.report.input_inference_eligible,
    "reason_codes": validated.report.reason_codes,
}
validation_summary

## 2. Versioned LR and NicheNet resources

Real analyses should load a checksum-pinned CellChatDB or CellPhoneDB bundle and a derived NicheNet target prior. The synthetic branch below creates the same typed contracts in memory only so this tutorial remains fast and portable. Synthetic resources are development fixtures, not biological references.


In [ ]:
def tiny_interaction(interaction_id: str, ligand: str, receptor: str) -> Interaction:
    return Interaction(
        interaction_id=interaction_id,
        source_interaction_id=interaction_id,
        ligand_name=ligand,
        receptor_name=receptor,
        ligand_subunits=(ligand,),
        receptor_subunits=(receptor,),
        ligand_is_complex=False,
        receptor_is_complex=False,
        direction="Ligand-Receptor",
        source="tutorial_fixture",
        version="1",
        species=Species.HUMAN,
        gene_namespace=GeneNamespace.HGNC_SYMBOL,
    )


def tiny_resources() -> tuple[ResourceBundle, TargetPrior]:
    bundle = ResourceBundle(
        resource_id="tutorial_lr",
        version="1",
        species=Species.HUMAN,
        gene_namespace=GeneNamespace.HGNC_SYMBOL,
        interactions=(
            tiny_interaction("i1", "L1", "R1"),
            tiny_interaction("i2", "L2", "R2"),
        ),
        mapping_report=MappingReport(2, 2, 4),
        manifest_digest="c" * 64,
        source_files=("tutorial-fixture.tsv",),
        license="CC0-1.0",
        citation="Synthetic tutorial fixture; not a biological reference.",
    )
    prior = TargetPrior(
        resource_id="tutorial_prior",
        version="1",
        species=Species.HUMAN,
        gene_namespace=GeneNamespace.HGNC_SYMBOL,
        driver_kind="interaction",
        target_ids=("T1", "T2"),
        driver_ids=("i1", "i2"),
        indptr=(0, 1, 2),
        target_indices=(0, 1),
        weights=(1.0, 1.0),
        ranks=None,
        direction=1,
        evidence="synthetic_tutorial_fixture",
        mapping_report=MappingReport(2, 2, 4),
        manifest_digest="d" * 64,
    )
    return bundle, prior


if USE_SYNTHETIC:
    lr_resource, target_prior = tiny_resources()
else:
    lr_resource = (
        crychic.load_cellchat_resource(DATABASE_ROOT, Species.HUMAN)
        if LR_RESOURCE == "cellchat"
        else crychic.load_cellphonedb_resource(DATABASE_ROOT)
    )
    target_prior = crychic.load_nichenet_target_prior(DATABASE_ROOT)

resource_summary = {
    "lr_resource": (lr_resource.resource_id, lr_resource.version),
    "n_interactions": len(lr_resource.interactions),
    "prior": (target_prior.resource_id, target_prior.version),
    "prior_driver_kind": target_prior.driver_kind,
}
resource_summary

## 3. Contrast, penalty tuning, and fold-learned nuisance

`ContrastSpec` defines the estimand over condition/group nodes. Add more balanced contrasts for multi-arm data, for example treatment A versus control and treatment B versus control. `PenaltyTuningSpec` pre-registers a deterministic inner subject-fold grid; it does not create a machine-learning training set. `FrozenLatentNuisanceSpec` learns a small receiver-autonomous basis only from each physical training split. Controls exclude every target in the complete frozen prior universe, including targets of currently ineligible families. Held-out subjects never enter factor fitting. This enables the opt-in descriptive OOF chain; it does not authorize p/q values, communication probabilities, or the default-method switch.


In [ ]:
contrast = balanced_contrast(
    positive=("stim",),
    negative=("control",),
    name="stim_vs_control",
)
reverse_contrast = balanced_contrast(
    positive=("control",),
    negative=("stim",),
    name="control_vs_stim",
)
directional_pair = DirectionalContrastPairSpec(
    forward_contrast=contrast,
    reverse_contrast=reverse_contrast,
)
tuning = PenaltyTuningSpec(
    lambda1_fractions=(1.0,),
    lambda2_fractions=(0.0,),
    inner_allowed_n_splits=(2,),
    root_seed=20260715,
)
latent_nuisance = crychic.FrozenLatentNuisanceSpec(
    max_components=1,
    min_control_features=2,
    min_training_subjects=2,
    minimum_explained_fraction=0.01,
)
crossfit_spec = crychic.CrossFitSpec(
    contrasts=(contrast, reverse_contrast),
    directional_pairs=(directional_pair,),
    training_spec=crychic.FoldTrainingSpec(
        min_cells=1 if USE_SYNTHETIC else 10,
        max_interactions=2 if USE_SYNTHETIC else None,
        sender_parameters=ContrastCommonSenderParameters(min_subjects=2),
    ),
    allowed_n_splits=(2,),
    min_train_subjects_per_context=2,
    min_test_subjects_per_context=1,
    latent_nuisance_spec=latent_nuisance,
    penalty_tuning_spec=tuning,
)
crossfit_spec.to_dict()

## 4. Run and persist `Crychic.fit_crossfit`

The output directory is atomic and versioned. Reusing an existing destination is rejected; this cell removes only this tutorial's own output when `OVERWRITE_OUTPUT=True`. Keep the producer-owned `CrossFitArtifacts` in memory for post-fit signatures and hypergraphs, then persist a separate `CrossFitResult` whose loader validates table hashes, schemas, application coverage, and descriptive-only claim fields.


In [ ]:
result_dir = OUTPUT_ROOT / "crossfit_result"
if OVERWRITE_OUTPUT and result_dir.exists():
    shutil.rmtree(result_dir)

model = crychic.Crychic(
    config,
    resource_bundle=lr_resource,
    target_prior=target_prior,
)
artifacts = model.fit_crossfit(
    adata,
    spec=crossfit_spec,
)
assert isinstance(artifacts, crychic.CrossFitArtifacts)
result = crychic.write_crossfit_result(artifacts, result_dir)
assert isinstance(result, crychic.CrossFitResult)
result = crychic.CrossFitResult.load(result_dir)
{
    "crossfit_result_id": result.manifest["crossfit_result_id"],
    "complete_pipeline_oof_certified": result.manifest[
        "complete_pipeline_oof_certified"
    ],
    "formal_inference_status": result.manifest["formal_inference_status"],
    "claim_scope": result.manifest["claim_scope"],
}

## 5. Query component ledger and descriptive differential

The component ledger has explicit grain. `component_scope='family'` describes receiver-family quantities; `lr_member` adds `interaction_id`; `sender_lr_member` adds `sender`. `condition` is represented by `context_id` plus the named contrast. Values from different receivers must not be row-union ranked as though they came from one global functional. The differential table contains held-out subject-family effects for within-receiver inspection and repeated diagnostics.


In [ ]:
component_ledger = result.read_components()
descriptive_differential = result.read_descriptive_differential()
forbidden_inference_fields = {
    "p",
    "p_value",
    "q",
    "q_value",
    "fdr",
    "posterior",
    "comm_probability",
    "confidence_interval",
    "standard_error",
}
assert forbidden_inference_fields.isdisjoint(component_ledger.columns)
assert forbidden_inference_fields.isdisjoint(descriptive_differential.columns)
assert not component_ledger["is_oof_certified"].any()
assert not descriptive_differential["is_oof_certified"].any()
print("component scopes:", component_ledger["component_scope"].value_counts().to_dict())
print(
    "differential status:", descriptive_differential["status"].value_counts().to_dict()
)
component_ledger.head()

In [ ]:
# These helpers retain observed, structural-zero, and not-estimable rows by default.
family_view = result.query_family_scores(contrast="stim_vs_control", mode="state")
lr_view = result.query_integrated_lr_scores(contrast="stim_vs_control", mode="state")
lr_component_audit = result.query_lr_pairs(contrast="stim_vs_control", mode="state")
sender_view = result.query_sender_lr_pairs(contrast="stim_vs_control", mode="state")
observed_sender_view = result.query_sender_lr_pairs(
    contrast="stim_vs_control", mode="state", status="observed"
)
subject_effect_view = descriptive_differential.loc[
    :,
    [
        "contrast",
        "receiver",
        "subject_id",
        "family_id",
        "differential_effect",
        "status",
        "reason_code",
    ],
]
print(
    "family rows",
    len(family_view),
    "LR rows",
    len(lr_view),
    "sender-LR rows",
    len(sender_view),
)
print("observed sender-LR rows", len(observed_sender_view))
subject_effect_view.head()

### Interpreting the columns

- **condition/group**: use `context_id` and the contrast definition together; the persisted table intentionally does not infer meaning from a display label alone.
- **receiver**: the cell population whose receptor eligibility and downstream response are modeled. Comparability is currently within receiver across contexts, not a global ranking across receivers.
- **family**: an attribution family on the frozen target-prior basis. Family rows may be observed, structural zero, or not estimable.
- **LR member**: `interaction_id` identifies a ligand-receptor member allocated within its family; inspect receptor and ligand gate status before interpreting a zero.
- **sender-LR member**: adds the candidate sender allocation. This is evidence allocation, not a causal sender assertion.
- **status/reason_code**: always interpret these before `component_value`. A missing or unsupported quantity is not a biological zero.


### 5.1 Descriptive fitted signatures and communication hypergraph

These views consume the live producer-owned `CrossFitArtifacts` and do not refit any numerical model. Signatures retain receiver-context, LR-attributed, and sender-LR-receiver layers plus an availability audit. The hypergraph uses subject-equal sender-resolved strength and explicit complex membership. Both are descriptive: no p/q values or calibrated communication probabilities are released, and a persisted `CrossFitResult` cannot be substituted for `artifacts`.


In [ ]:
signature_export = model.export_crossfit_signatures(
    artifacts, mode="state", entropy_threshold=0.8
)
hypergraph = model.export_crossfit_hypergraph(artifacts)
assert signature_export.inference_eligible is False
assert hypergraph.inference_eligible is False
signature_availability = signature_export.availability_table()
hyperedges = hypergraph.hyperedges
assert forbidden_inference_fields.isdisjoint(signature_availability.columns)
assert forbidden_inference_fields.isdisjoint(hyperedges.columns)
print(
    "signature availability:", signature_availability["status"].value_counts().to_dict()
)
print("hyperedges:", hyperedges["status"].value_counts().to_dict())

### 5.2 Common-scale signed target programs

The explicit directional pair also enables a source-agnostic target-program table. The universe is frozen from the exact cross-fit TargetPrior and feature order without outcome values. The producer estimates one signed subject-equal OOF receiver gene effect on the shared `log1p_cpm` scale, then compares its positive and negative channels with every frozen program. Reverse means reduced activation-compatible evidence, not active inhibition. These rows contain no p/q/probability fields and are not LR-edge or native NicheNet results.


In [ ]:
program_universe = crychic.freeze_crossfit_directional_target_program_universe(
    artifacts
)
signed_programs = crychic.score_crossfit_directional_target_programs(
    artifacts,
    program_universe,
    pair_spec_id=directional_pair.pair_spec_id,
)
signed_program_table = signed_programs.scores
assert len(signed_program_table) == (
    len(signed_programs.receiver_ids)
    * len(signed_programs.program_ids)
    * len(signed_programs.channels)
)
assert set(signed_program_table["sender"]) == {"__source_agnostic__"}
assert forbidden_inference_fields.isdisjoint(signed_program_table.columns)
signed_program_table.head()

## 6. Repeated cross-fit split-stability

`RepeatedCrossFitSpec` reruns the complete train/apply chain under distinct subject partitions. Its outputs quantify split stability and selection opportunity; they are not confidence intervals or formal hypothesis tests. Set `RUN_REPEATED=True` to execute two repeats on the tiny fixture.


In [ ]:
repeated_spec = crychic.RepeatedCrossFitSpec(
    crossfit_spec=crossfit_spec,
    n_repeats=2,
)
if RUN_REPEATED:
    repeated = model.fit_repeated_crossfit(adata, spec=repeated_spec)
    repeat_registry = repeated.repeat_registry
    family_fold_events = repeated.family_fold_events
    subject_family_repeat_values = repeated.subject_family_repeat_values
    family_repeat_stability = repeated.family_repeat_stability
    assert not repeated.is_inference_eligible
    print(repeated.diagnostic_status, repeated.formal_inference_status)
    print(repeat_registry.to_string(index=False))
    print(family_repeat_stability.head().to_string(index=False))
else:
    print("Set RUN_REPEATED=True to run two full subject cross-fit repeats.")

## 7. Small full-pipeline resampling diagnostic

`resample_crossfit` materializes legal subject bootstrap or context-permutation inputs and reruns the full cross-fit chain. A tiny `1 + 1` run is useful for checking exchangeability and execution records, but it is far below calibration requirements. The current result manifest explicitly reports `formal_inference_status='not_released_full_pipeline_resampling_diagnostic_only'`, `is_inference_eligible=False`, and no inferential fields.


In [ ]:
if RUN_RESAMPLING:
    resampling = model.resample_crossfit(
        adata,
        spec=crossfit_spec,
        n_bootstraps=1,
        n_permutations=1,
        retain_children=False,
    )
    resampling_manifest = resampling.to_manifest()
    assert resampling_manifest["full_pipeline_refit_per_resample"] is True
    assert resampling_manifest["score_table_relabeling_only"] is False
    assert resampling_manifest["is_inference_eligible"] is False
    assert resampling_manifest["inferential_fields_available"] == []
    resample_records = pd.DataFrame(resampling_manifest["records"])
    print(
        resample_records[
            ["operation", "resample_index", "status", "failure_code"]
        ].to_string(index=False)
    )
else:
    print(
        "Set RUN_RESAMPLING=True for one bootstrap and one legal "
        "permutation diagnostic."
    )

## 8. Basic publication-quality descriptive figure

This figure summarizes observed subject-family differential effects within each receiver. It deliberately shows individual subjects and a zero reference line, does not draw inferential error bars, and labels the output as descriptive. Save vector PDF plus high-resolution PNG for downstream figure assembly.


In [ ]:
plot_data = descriptive_differential.loc[
    descriptive_differential["status"].eq("observed"),
    ["receiver", "family_id", "subject_id", "differential_effect"],
].copy()
if plot_data.empty:
    print(
        "No observed differential rows in this fixture; inspect reason_code "
        "before plotting zeros."
    )
else:
    top_families = (
        plot_data.groupby(["receiver", "family_id"], observed=True)[
            "differential_effect"
        ]
        .apply(lambda values: float(np.mean(np.abs(values))))
        .sort_values(ascending=False)
        .head(12)
        .index
    )
    selected = (
        plot_data.set_index(["receiver", "family_id"]).loc[top_families].reset_index()
    )
    selected["receiver_family"] = (
        selected["receiver"] + " | " + selected["family_id"].str.slice(0, 12)
    )
    sns.set_theme(context="paper", style="ticks", font_scale=1.0)
    fig, ax = plt.subplots(
        figsize=(7.2, max(3.2, 0.34 * selected["receiver_family"].nunique()))
    )
    sns.stripplot(
        data=selected,
        x="differential_effect",
        y="receiver_family",
        hue="receiver",
        dodge=False,
        alpha=0.72,
        size=4.5,
        ax=ax,
    )
    ax.axvline(0.0, color="#222222", linewidth=0.8, linestyle="--")
    ax.set_xlabel("Held-out subject-family differential effect")
    ax.set_ylabel("Receiver | family")
    ax.set_title("CRYCHIC descriptive cross-fit effects")
    ax.legend(
        title="Receiver", frameon=False, bbox_to_anchor=(1.02, 1), loc="upper left"
    )
    sns.despine(ax=ax)
    fig.tight_layout()
    fig.savefig(
        OUTPUT_ROOT / "descriptive_subject_family_effects.pdf", bbox_inches="tight"
    )
    fig.savefig(
        OUTPUT_ROOT / "descriptive_subject_family_effects.png",
        dpi=400,
        bbox_inches="tight",
    )
    plt.show()

## Developer handoff checklist

Before interpreting a real cohort, verify: sample-to-subject mapping is one-to-many only in the intended direction; every condition/group has enough independent subjects; resource species and gene namespace match the H5AD; no interaction cap was introduced for a primary benchmark; contrasts and penalty grids were declared before inspecting results; receiver-specific rows are not pooled into a global rank; `status` and `reason_code` are propagated; and all formal p/q/probability fields remain absent until the calibrated inference release. The signed target-program table remains descriptive; a frozen multi-seed Track B release evaluation and cross-method benchmarking are separate workflows and cannot be inferred from this notebook.
